# Postgres PGVectorStore

### DataBase connection

In [8]:
# @title Set your values or use the defaults to connect to Docker { display-mode: "form" }
POSTGRES_USER = "test"  # @param {type: "string"}
POSTGRES_PASSWORD = "test"  # @param {type: "string"}
POSTGRES_HOST = "127.0.0.1"  # @param {type: "string"}
POSTGRES_PORT = "6024"  # @param {type: "string"}
POSTGRES_DB = "vectorDB"  # @param {type: "string"}
TABLE_NAME = "test"  # @param {type: "string"}
VECTOR_SIZE = 2000  # @param {type: "int"} Max vector size for PGVector

In [9]:
# See docker command above to launch a Postgres instance with pgvector enabled.
CONNECTION_STRING = (
    f"postgresql+asyncpg://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}"
    f":{POSTGRES_PORT}/{POSTGRES_DB}"
)
# To use psycopg3 driver, set your connection string to `postgresql+psycopg://`

In [10]:
import os
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    openai_api_key=os.getenv("OPENAI_API_KEY")
)

In [11]:
from langchain_postgres import PGEngine

pg_engine = PGEngine.from_connection_string(url=CONNECTION_STRING)

#### Create vector database

In [12]:
from langchain_postgres import Column

await pg_engine.ainit_vectorstore_table(
    table_name=TABLE_NAME,
    vector_size=VECTOR_SIZE,
    id_column=Column(name='id', data_type='INTEGER'),
    overwrite_existing=True,
)
print(f"Table '{TABLE_NAME}' created successfully.")

Table 'test' created successfully.


#### OpenAI Embeddings

#### Vector storage

In [13]:
from langchain_postgres import PGVectorStore

vector_store = PGVectorStore.create_sync(
    engine=pg_engine,
    table_name=TABLE_NAME,
    embedding_service=embeddings,
    id_column='id',
)

#### Vector indexes

In [14]:
from langchain_postgres.v2.indexes import HNSWIndex, IVFFlatIndex

# index = IVFFlatIndex()
# await vector_store.aapply_vector_index(index)

index = HNSWIndex()
await vector_store.aapply_vector_index(index)